# Session log — LLM-in-Sandbox on 2× T4

The **actual** session behind the repo, with real outputs, kept because the dead
ends are the useful part: several of them changed the design.

For a clean re-run use `colab_driver.ipynb`. This is a record, not a recipe.

| # | probe | finding |
|---|---|---|
| 1 | `nvidia-smi` | 2× T4, **sm75** — fp16 only, no FA2/FP8/Marlin, `PHB` (no NVLink) |
| 2 | serve @ 32k | KV cache needs 4.50 GiB, 4.07 GiB free → context capped at 24576 |
| 3 | `transformers` | Colab ships 5.0; vLLM 0.11 calls the removed `all_special_tokens_extended` |
| 4 | PTY trials | bash spawns fine → `shell died (exit -9)` was **our** bug |
| 5 | raw PTY replay | sentinel *does* arrive → we were scanning the wrong buffer |
| 6 | `unshare` | **denied** on Colab, even as uid 0 |
| 7 | seccomp | **permitted** → enforced network ablation recovered |
| 8 | attention backend | FlexAttention 22 tok/s → TRITON_ATTN 51–69 tok/s per card |
| 9 | first real sweep | four measurement bugs, three biasing toward the paper's result |

## 1. What hardware did we actually get?

`30 GB of VRAM` turns out to be **2×15 GB at sm75**, not one big card. That reframes everything downstream.

In [1]:
import subprocess, sys, os, shutil, platform
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version,compute_cap","--format=csv"],capture_output=True,text=True).stdout)
print("python", sys.version.split()[0], "| cpus", os.cpu_count())
print(subprocess.run(["free","-g"],capture_output=True,text=True).stdout)
import torch; print("torch", torch.__version__, "cuda", torch.version.cuda, "bf16", torch.cuda.is_bf16_supported())
for t in ["docker","bwrap","unshare","runsc"]:
    print(f"{t}: {shutil.which(t)}")
print("uid", os.getuid())

name, memory.total [MiB], driver_version, compute_cap
Tesla T4, 15360 MiB, 580.159.04, 7.5
Tesla T4, 15360 MiB, 580.159.04, 7.5

python 3.12.13 | cpus 4
               total        used        free      shared  buff/cache   available
Mem:              31           1          26           0           3          29
Swap:              0           0           0

torch 2.10.0+cu128 cuda 12.8 bf16 True
docker: None
bwrap: None
unshare: /usr/bin/unshare
runsc: None
uid 0

Two things to flag:

- `is_bf16_supported()` returns **True** on a T4 and is misleading — sm75 has no
  native bf16 path.
- No `docker`, no `bwrap`. But `unshare` exists and we are **uid 0**, which
  looked promising for isolation (see §6 for how that turned out).

In [2]:
for host,port in [("pypi.org",443),("huggingface.co",443),("github.com",443)]:
    try:
        socket.create_connection((host,port), timeout=8); print(f"NET  {host}:{port} OK")
    except Exception as e: print(f"NET  {host}:{port} FAIL {e}")
for m in ["vllm","transformers","datasets","openai","flash_attn","xformers"]:
    print(f"PKG  {m}", md.version(m) if importlib.util.find_spec(m) else "MISSING")
print("P2P ", sh("nvidia-smi topo -m | head -5"))

NET  pypi.org:443 OK
NET  huggingface.co:443 OK
NET  github.com:443 OK
PKG  vllm MISSING
PKG  transformers == 5.0.0
PKG  datasets == 4.8.5
PKG  openai == 2.31.0
PKG  flash_attn MISSING
PKG  xformers MISSING
P2P       GPU0    GPU1    CPU Affinity    NUMA Affinity
GPU0     X      PHB     0-3     0
GPU1    PHB      X      0-3     0

**`PHB`** = PCIe host bridge, no NVLink. That is what later rules out tensor-parallel in favour of one replica per GPU. Note `transformers 5.0.0` — it bites in §4.

## 2. Install vLLM — detached

`nohup ... &` because the Colab MCP bridge has **no kernel interrupt**: a blocking cell wedges the kernel for everything else.

In [3]:
open("/content/install_vllm.sh","w").write(textwrap.dedent("""
pip install "vllm==0.11.0" 2>&1 | tail -40
python -c "import torch, vllm; print('INSTALL_OK', vllm.__version__, torch.__version__)"
echo "___INSTALL_DONE___ rc=$?"
"""))
subprocess.Popen("nohup bash /content/install_vllm.sh > /content/logs/install_vllm.log 2>&1 &", shell=True)
print("launched; poll /content/logs/install_vllm.log")

launched; poll /content/logs/install_vllm.log

In [5]:
print(subprocess.run("tail -c 1500 /content/logs/install_vllm.log", shell=True,
                     capture_output=True, text=True).stdout)

tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
gradio 5.50.0 requires starlette<1.0,>=0.40.0, but you have starlette 1.6.0 which is incompatible.
  triton-3.4.0 vllm-0.11.0 xformers-0.0.32.post1 xgrammar-0.1.25
+ python -
INSTALL_OK vllm 0.11.0 torch 2.8.0+cu128 cuda 12.8
device Tesla T4 (7, 5)
___INSTALL_DONE___ rc=0

DONE

Installing vLLM **downgraded torch 2.10 → 2.8.0+cu128** and pulled in xformers. Device confirmed as `(7, 5)`.

## 3. Two serving failures, in order

**(a) KV cache does not fit at 32k.**

In [13]:
print(subprocess.run("grep -E 'ERROR|ValueError' /content/logs/vllm_4b.log | tail", shell=True,
                     capture_output=True, text=True).stdout)

ERROR [core.py:708] ValueError: To serve at least one request with the models's max seq len
 (32768), (4.50 GiB KV cache is needed, which is larger than the available KV cache memory
 (4.07 GiB). Based on the available memory, the estimated maximum model length is 29600.
 Try increasing `gpu_memory_utilization` or decreasing `max_model_len`.
RuntimeError: Engine core initialization failed.
___SERVER_EXITED___ rc=1

**(b) `transformers` 5.0 removed an attribute vLLM still calls.**

```
AttributeError: Qwen2Tokenizer has no attribute all_special_tokens_extended
```

vLLM's pin is `>=4.55.2`, which 5.0 satisfies, so pip never downgraded it. Fix:
`transformers==4.56.2` and `--max-model-len 24576`.

## 4. The shell bug: `shell died (exit -9)`

All 16 shell tests failed on Colab. `-9` is SIGKILL. First question: the platform, or us?

In [9]:
print(sh("cd /content/llm-in-sandbox && python -m pytest tests/ -q 2>&1 | tail -20"))

FAILED tests/test_shell.py::test_basic_command_and_exit_code - sandbox_lab.sa...
FAILED tests/test_shell.py::test_state_persists_across_calls - sandbox_lab.sa...
FAILED tests/test_shell.py::test_command_with_heredoc_survives_framing - sand...
FAILED tests/test_shell.py::test_timeout_interrupts_command_but_shell_survives
... (16 failed)

E   sandbox_lab.sandbox.shell.ShellError: shell died (exit -9)

Spawn bash exactly as `PersistentShell` does, four ways:

In [10]:
def trial(name, use_setsid, use_ctty):
    m, s = pty.openpty()
    a = termios.tcgetattr(s); a[3] &= ~termios.ECHO; a[1] &= ~termios.ONLCR
    termios.tcsetattr(s, termios.TCSANOW, a)
    def pre():
        if use_setsid: os.setsid()
        if use_ctty: fcntl.ioctl(s, termios.TIOCSCTTY, 0)
    p = subprocess.Popen(["bash","--noprofile","--norc"], stdin=s, stdout=s, stderr=s,
                         preexec_fn=pre, close_fds=True, cwd="/tmp")
    os.close(s); time.sleep(1.0); os.set_blocking(m, False)
    print(f"{name}: poll={p.poll()} out={os.read(m, 4096)[:80]!r}")

trial("A: setsid+ctty", True,  True)
trial("B: setsid only", True,  False)
trial("C: neither    ", False, False)
trial("D: ctty only  ", False, True)

A: setsid+ctty  : poll=None (rc -9 = SIGKILL) out=b'\x1b[?2004hbash-5.1# '
   after echo: b'\x1b[?2004l\rALIVE_OK\n\x1b[?2004hbash-5.1# '
B: setsid only  : poll=None (rc -9 = SIGKILL) out=b'\x1b[?2004hbash-5.1# '
   after echo: b'\x1b[?2004l\rALIVE_OK\n\x1b[?2004hbash-5.1# '
C: neither      : poll=None out=b'bash: cannot set terminal process group (103): Inappropriate ioctl for device\nbash: no job control in this shell\n'
   after echo: b'ALIVE_OK\n'
D: ctty only    : POPEN RAISED SubprocessError: Exception occurred in preexec_fn.

`poll=None` in every viable variant — **bash is alive**. So the SIGKILL was
ours. Also visible: `\x1b[?2004h`, readline's bracketed-paste escapes, which
would land in the model's observations (fixed later with `--noediting`), and
that `setsid` is a prerequisite for `TIOCSCTTY` (trial D).

Next: replay the exact init sequence and watch raw PTY bytes.

In [12]:
drain("after spawn")
os.write(m, b"set -m\nunset HISTFILE\nexport PS1= PS2=\n")
drain("after init block")
open("/tmp/.sandbox_lab/cmd_1.sh","w").write(":")
os.write(m, b"source '/tmp/.sandbox_lab/cmd_1.sh'\n")
drain("after source")
os.write(m, b"printf '\\n__SENT_%d__\\n' \"$?\"\n")
drain("after printf")
os.write(m, b"echo hello\n")
drain("after echo hello")

[after spawn] alive=True b'\x1b[?2004hbash-5.1# '
[after init block] alive=True b'\x1b[?2004l\r\x1b[?2004hbash-5.1# \x1b[?2004l\r'
[after source] alive=True b''
[after printf] alive=True b'\n__SENT_0__\n'
[after echo hello] alive=True b'hello\n'
final alive: True

**The sentinel arrives.** So the framing was right and the read loop was
looking in the wrong place. Two compounding bugs:

1. `_pump` scanned `_HeadTailBuffer._tail`, but that buffer fills **head-first**
   (32 KB). For any normal command the tail is empty, the sentinel never
   matched, and *every* command timed out.
2. On timeout `_interrupt` escalated with `killpg(getpgid(bash_pid))` — but
   after `setsid()` bash **is** its own group leader, so it SIGKILLed the shell
   it was meant to preserve. Hence `-9`.

Fixes: an 8 KB sliding window scanned for the sentinel, and `os.tcgetpgrp()` to
read the *foreground* group from the terminal so the command is signalled and
bash is not.

In [14]:
print(sh("cd /content/llm-in-sandbox && python -m pytest tests/ -q --no-header 2>&1 | tail -6"))

................F..............                                          [100%]
FAILED tests/test_shell.py::test_nonzero_exit_code_is_reported
E       AssertionError: assert None == 3
E        + where None = ShellResult(output='', exit_code=None, ...).exit_code

30/31. The last failure was **my test being wrong**: `exit 3` inside a
*sourced* file exits the shell itself — correct terminal semantics. So `exit`
now sets a `shell_died` flag (distinguishable from a timeout, which also yields
`exit_code=None`) and `Sandbox` rebuilds the session and tells the model its
shell state was reset, rather than every later command failing inexplicably.

In [16]:
print(sh("cd /content/llm-in-sandbox && python -m pytest tests/ -q --no-header 2>&1 | tail -4"))

..........................ss.....................                        [100%]

## 5. Enforcing ablations: `unshare` says no

In [17]:
for cmd in ["unshare --fork --pid --mount true",
            "unshare --fork --pid --mount --net true",
            "unshare --net ip link set lo up"]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    print(f"rc={r.returncode:3d}  {cmd}   {r.stderr.strip()[:60]}")

SKIPPED [1] tests/test_sandbox.py:119: unshare unavailable (needs uid 0 or unprivileged userns)
SKIPPED [1] tests/test_sandbox.py:144: unshare unavailable

=== direct unshare probes ===
rc=  1  unshare --fork --pid --mount true          unshare failed: Operation not permitted
rc=  1  unshare --fork --pid --mount --net true    unshare failed: Operation not permitted
rc=  1  unshare --net ip link set lo up            unshare failed: Operation not permitted

Denied even as **uid 0** — the container drops `CAP_SYS_ADMIN`. Namespace-based ablation is off the table on Colab.

## 6. …but seccomp says yes

An unprivileged process may install a seccomp-BPF filter after setting `PR_SET_NO_NEW_PRIVS`. This probe rescued the ablation design.

In [25]:
prog = (F*9)(F(LD,0,0,4), F(JEQ,0,5,0xC000003E), F(LD,0,0,0), F(JEQ,0,3,41),
             F(LD,0,0,16), F(JEQ,2,0,2), F(JEQ,1,0,10),
             F(RET,0,0,ALLOW), F(RET,0,0,DENY))
libc = ctypes.CDLL(ctypes.util.find_library("c"), use_errno=True)
assert libc.prctl(38,1,0,0,0) == 0                      # PR_SET_NO_NEW_PRIVS
assert libc.syscall(317,1,0,ctypes.byref(fprog)) == 0   # seccomp(SET_MODE_FILTER)
print("SECCOMP_INSTALLED")
try: socket.create_connection(("1.1.1.1",443),timeout=4); print("NET: REACHED (bad)")
except Exception as e: print("NET: blocked ->", type(e).__name__, e)
s = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM); s.close(); print("UNIX socket: still OK")
print("file write:", (open("/tmp/_sec.txt","w").write("x"), "OK")[1])

SECCOMP_INSTALLED
NET: blocked -> PermissionError [Errno 13] Permission denied
UNIX socket: still OK
file write: OK

Exactly the shape we want: **external** access gone, local IPC and files intact. This became `sandbox_lab/sandbox/seccomp.py`.

In [33]:
print(sh("cd /content/lis-test && PYTHONPATH=src python -m pytest tests/test_seccomp.py -q -rs"))

.......                                                                  [100%]

All 7 pass, each paired with a **positive control** — the same probe must
succeed with the capability enabled, because "the network is down" also passes
on a host with no network.

Two of them failed first for an instructive reason: with the network correctly
blocked, Python prints the failing **source line** in its traceback, so a
success marker spelled out in source (`print('REACHED')`) appeared in the output
of a correctly-blocked run. The marker is now assembled at runtime
(`'NET' + 'OK'`) so the literal never appears in source.

## 7. Throughput: the attention backend is the whole game

The first sweep decoded at ~22 tok/s aggregate with GPU1 **idle**. vLLM had auto-selected FlexAttention (FA2 needs sm80+), a `torch.compile` fallback rather than a real kernel.

In [19]:
print(subprocess.run("grep -E 'Avg generation throughput' /content/logs/vllm_qwen3-4b.log | tail -3",
                     shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv",
                     shell=True, capture_output=True, text=True).stdout)

Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 22.4 tokens/s,
  Running: 4 reqs, Waiting: 0 reqs, GPU KV cache usage: 10.1%, Prefix cache hit rate: 75.7%

index, utilization.gpu [%], memory.used [MiB]
0, 99 %, 14127 MiB
1, 0 %, 3 MiB

GPU0 pinned at 99%, **GPU1 at 0%**. Two levers: a real kernel, and the second card.

In [23]:
subprocess.run("pkill -f api_server; sleep 3", shell=True, capture_output=True)
launch(8000, 0); launch(8001, 1)   # VLLM_ATTENTION_BACKEND=TRITON_ATTN, one per GPU
for port in (8000, 8001):
    print(port, "UP" if up(port) else "not up", bench(port) if up(port) else "")

port 8000: UP   Using Triton backend
port 8001: UP   Using Triton backend
  port 8000: 707 tok in 13.7s = 51.5 tok/s aggregate (4 concurrent)
  port 8001: 797 tok in 11.6s = 68.9 tok/s aggregate (4 concurrent)

**~22 → ~120 tok/s total**: roughly 2.5–3× from the backend, 2× from using
the second card. One replica per GPU rather than TP=2, because `PHB` means no
NVLink and agent episodes parallelise across replicas for free.

## 8. End-to-end smoke test

Same question, both modes. Where the loop's real failure modes appeared.

In [27]:
Q = "What is the remainder when 7^{1234} is divided by 1000? Give the final integer."
for mode in ["sandbox", "direct"]:
    cfg = AgentConfig(model="qwen3-4b", mode=mode, max_turns=12, max_tokens_per_turn=1024)
    t = SandboxAgent(client, cfg).run(Q, task_id="smoke", backend="local")
    print(f"== {mode} == answer={t.final_answer!r} stop={t.stop_reason} "
          f"turns={t.n_turns} gen_tokens={t.generated_tokens}")

===== sandbox =====
answer: '849' | stop: stalled | turns: 3 | gen_tokens: 812
   turn0 -> bash({"command": "python3 -c \"print(pow(7, 1234, 1000))\""})
      obs: '849'
   turn1 content: "The remainder when $7^{1234}$ is divided by 1000 is 849.\n\nfinish(849)"
   turn2 content: 'finish(849)'

===== direct =====
answer: '849' | stop: answered | turns: 1 | gen_tokens: 1660
   turn0 content: '... x \\equiv 849 \\mod 1000 ... FINAL ANSWER: 849'

Both correct (849). Tokens: **812 vs 1660 = 0.49×**, at the bottom of the
paper's reported 0.49–0.84× band — but this is one item, so an anecdote.

Two harness bugs surfaced here, both biasing the sandbox arm downwards:

1. It solved the task on **turn 0**, then burned turns emitting no tool call.
   Fixed: nudge once, stop on the second consecutive miss.
2. It wrote **`finish(849)` as prose** instead of emitting a tool call.
   Recovered by a deliberately narrow parser (only `finish`, only trailing),
   tested against inventing a call from "I will call finish(x) once…".

The `direct` arm also returned `'FINAL ANSWER: 849'` — the label leaked into the
extracted answer and would have graded as wrong.

## 9. The first real sweep — and four measurement bugs

100 items (25 × 4 domains) per arm, resumable, 8 workers across 2 servers.

In [40]:
print(sh("tail -c 900 /content/logs/pilot.log"))
rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
for mode in ["direct","sandbox"]:
    sub = [r for r in rows if r["mode"]==mode]
    if sub: print(f"  {mode:8s} n={len(sub):3d} acc={sum(r['correct'] for r in sub)/len(sub):.3f} "
                  f"tok={sum(r['generated_tokens'] for r in sub)/len(sub):7.0f}")
print(collections.Counter(r["stop_reason"] for r in rows).most_common())

  75/96  acc=0.627  4.4/min  last=mmlu_pro/biomedicine/6493
  90/96  acc=0.611  5.0/min  last=mmlu_pro/chemistry/3952
  95/96  acc=0.589  3.6/min  last=mmlu_pro/chemistry/3622

rows: 100
  direct   n=100  acc=0.590  tok=    647  turns=0.9
      by domain: {'biomedicine': 25, 'chemistry': 25, 'math': 25, 'physics': 25}
  stop: [('answered', 76), ('no_answer_parsed', 17), ('error', 7)]

**23% of baseline episodes produced no usable answer.** That needed explaining before trusting anything.

In [42]:
bad = [r for r in rows if r["stop_reason"] in ("no_answer_parsed","error")]
print("errors:", collections.Counter(r["error"][:60] for r in rows if r.get("error")).most_common())
print("their token counts:", sorted(r["generated_tokens"] for r in rows if r["stop_reason"]=="no_answer_parsed"))
for r in bad[:2]:
    t = json.load((tdir / f"direct_full_{r['task_id'].replace('/','_')}.json").open())
    print("tail:", repr(t["turns"][0]["content"][-200:]))

errors: [('APITimeoutError: Request timed out.', 7)]

no_answer_parsed by domain: [('biomedicine', 14), ('physics', 2), ('math', 1)]
their token counts: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 6, 6, 6, 9, 11, 15, 15]

=== mmlu_pro/math/8781 gen=15 ===
...tail: 'F. The population of SAT scores from each group is normally distributed.'

=== mmlu_pro/physics/9873 gen=2 ===
...tail: 'C'

**A grader bug, not a model failure.** Two to fifteen tokens: the model
answered `C` and was scored as having produced *no answer*. It was obeying
MMLU-Pro's own instruction ("answer with the letter of the correct option
only"), which overrides the system prompt's `FINAL ANSWER:` format.

The sandbox arm answers through the `finish` tool and never touches that path —
so this bug was structurally incapable of affecting it. Left in, it would have
handed the sandbox a ~10 point advantage made entirely of formatting.

In [44]:
print(sh("cd /content/lis-test && PYTHONPATH=src python scripts/regrade.py "
         "/content/runs/pilot_qwen3-4b --dry-run"))

episodes: 132  changed: 22  no trajectory: 0
answer source: {'parsed': 94, 'none': 12, 'finish': 26}
grade flips: {'False->True': 11, 'False->False': 9, 'True->False': 2}
  direct   n=100 acc=0.700
  sandbox  n= 32 acc=0.625

dry run: nothing written

**Baseline accuracy 0.596 → 0.700.** But 2 episodes flipped True→False, which had to be understood before writing anything.

In [46]:
for r in rows:
    ans, src = rg.answer_from_trajectory(json.load(f.open()))
    if r["correct"] and not grade(ans, r["gold"], kind):
        print(f"REGRESSION {r['task_id']} mode={r['mode']} old={r['predicted']!r} new={ans!r}")
        print("   tool calls last turn:", t["turns"][-1]["tool_calls"])

REGRESSION mmlu_pro/physics/9157 mode=sandbox gold='C'
   old_pred='C'  new_pred=None (src=none) stop=max_turns
   last content tail: '...{"name": "bash", "arguments": {}}\n</tool_call>'
   tool calls last turn: [{'name': 'bash', 'arguments': {}}, {'name': 'bash', 'arguments': {}}]

REGRESSION mmlu_pro/physics/10089 mode=sandbox gold='I'
   old_pred='I'  new_pred=None (src=none) stop=max_turns
   last content tail: '<tool_call>\n{"name": "bash", "command": "python3 -c \"print((0.75 + 0.75) / (1 + 0.75 * 0.75))\""}\n</tool_call>'
   tool calls last turn: [{'name': 'bash', 'arguments': {}}]

This single output exposed **three more bugs**, all hurting the sandbox arm:

1. `{"name": "bash", "command": ...}` — the model put parameters at the **top
   level**, not nested under `"arguments"`. Reading only `arguments` gave `{}`,
   every command failed with "command is required", and the episode burned all
   20 turns. In a results table that is indistinguishable from a 4B model that
   cannot use tools — **exactly the paper's reported finding**. It would have
   looked like a successful reproduction.
2. The forced-answer call made after `max_turns` was not recorded as a turn, so
   its completion tokens went **uncounted**, understating sandbox token use.
3. `regrade.py` could overwrite a real answer with `None` for those episodes,
   since the forced answer existed only in the results row, not on disk.

All 42 sandbox episodes were deleted and the arm restarted under fixed code:
they were harness artefacts, not model behaviour.

## 10. What this sample size can actually resolve

Before reading any delta: exercise `report.py` on a synthetic set with a **genuine +8pp effect** built in, at the same n.

In [48]:
# 60 paired items, true effect: baseline p=0.60, sandbox p=0.68
print(subprocess.run([sys.executable,"scripts/report.py",str(d/"results.jsonl")],
                     capture_output=True,text=True).stdout)

| domain | n | baseline | sandbox | delta (pp) | 95% CI | McNemar p |
|---|---:|---:|---:|---:|---:|---:|
| biomedicine | 15 | 60.0 | 86.7 | +26.7 | [0.0, 53.3] | 0.219 |
| chemistry | 15 | 53.3 | 60.0 | +6.7 | [-33.3, 46.7] | 1.000 |
| math | 15 | 53.3 | 46.7 | -6.7 | [-26.7, 13.3] | 1.000 |
| physics | 15 | 40.0 | 66.7 | +26.7 | [-6.7, 60.0] | 0.289 |
| ALL | 60 | 51.7 | 65.0 | +13.3 | [-3.3, 30.0] | 0.185 |

| domain | n | baseline | sandbox | ratio | 95% CI |
| ALL | 60 | 662 | 330 | 0.50x | [0.46, 0.54] |

A **real** +8pp effect shows up as +13.3pp with a 95% CI of
`[-3.3, +30.0]` and p=0.185 — **not significant**. Per-domain it is noise:
+26.7, +6.7, −6.7, +26.7.

So at 25 items/domain a null is the expected outcome, and the accuracy delta
must be reported with its CI rather than as a point estimate. The **token
ratio** is far better resolved (`[0.46, 0.54]`) because it is a continuous
measure — which is why the efficiency claim is the one this setup can actually
test.

Detecting the paper's ~5–15pp accuracy effects needs several hundred items per
domain: many hours on a T4.

## Status at session end

`direct` arm complete (100/100). `sandbox` arm restarted from zero under fixed
code and still running. See `docs/RESUME.md` for exact resume commands and the
one ordering trap: `regrade.py` **rewrites** `results.jsonl` while the sweep
**appends** to it, so it must run only after the sweep exits.